# S04 — Phase 2 checkpoint-selection robustness (VAL only)

This focused audit reproduces the study-specific validation heuristic, checks Pareto non-dominance, and evaluates the stability of the residual-only versus temporal variant decision under parameter perturbations. TEST data are not used.

In [ ]:
from pathlib import Path
import json
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

ROOT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
OUT_DIR = ROOT / "Results" / "Phase2_Checkpoint_Selection_Robustness_VAL_Only"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

RUNS = {
    ("Ifran", "residual_only"): ROOT / "Ablations" / "Phase2_D_K_Lambda_Delta_VAL_Only" / "runs" / "ifran" / "IFRAN_B4_C15_CTRL_GL000_D5_K2_HD3_SEED42",
    ("Ifran", "temporal"): ROOT / "Ablations" / "Phase2_D_K_Lambda_Delta_VAL_Only" / "runs" / "ifran" / "IFRAN_B4_C15_D5_K2_GL020_HD3_SEED42",
    ("Maamoura", "residual_only"): ROOT / "Ablations" / "Phase2_D_K_Lambda_Delta_VAL_Only" / "runs" / "maamoura" / "MAAMOURA_B4_C15_CTRL_GL000_D2_K2_HD3_SEED42",
    ("Maamoura", "temporal"): ROOT / "Ablations" / "Phase2_D_K_Lambda_Delta_VAL_Only" / "runs" / "maamoura" / "MAAMOURA_B4_C15_D2_K2_GL005_HD3_SEED42",
    ("Agadir", "residual_only"): ROOT / "Ablations" / "Phase2_D_K_Lambda_Delta_VAL_Only" / "runs" / "agadir" / "AGADIR_B4_C15_CTRL_GL000_D3_K3_HD3_SEED42",
    ("Agadir", "temporal"): ROOT / "Ablations" / "Phase2_D_K_Lambda_Delta_VAL_Only" / "runs" / "agadir" / "AGADIR_B4_C15_D3_K3_GL010_HD3_SEED42",
}

COMMON_SUPPORT_METRICS = ROOT / "Results" / "Final_Phase_Selection_VAL_Only" / "tables" / "02_threeway_exact_paired_val_metrics.csv"

BASE = {
    "w_rmse": 0.25,
    "w_slope": 2.00,
    "w_sd": 0.75,
    "w_bias": 0.20,
    "w_r2": 0.50,
    "slope_floor": 0.90,
    "r2_floor": 0.70,
    "bias_cap": 2.00,
}

RANDOM_SEED = 42
N_SENSITIVITY = 5000

print(f"ROOT: {ROOT}", flush=True)
print(f"Outputs: {OUT_DIR}", flush=True)

In [ ]:
def assert_val_only_path(path: Path) -> None:
    if re.search(r"test", str(path), flags=re.IGNORECASE):
        raise AssertionError(f"TEST-like path is forbidden in this notebook: {path}")


def assert_no_test_columns(df: pd.DataFrame, source: str) -> None:
    bad = [c for c in df.columns if re.search(r"test", str(c), flags=re.IGNORECASE)]
    if bad:
        raise AssertionError(f"TEST-like columns found in {source}: {bad}")


for path in [*RUNS.values(), COMMON_SUPPORT_METRICS]:
    assert_val_only_path(path)

required_history = {
    "step", "val_n", "val_mae", "val_rmse", "val_r2", "val_bias",
    "val_slope", "val_std_ratio", "val_article_compromise_score",
}

preflight_rows = []
for (site, variant), run_dir in RUNS.items():
    history_path = run_dir / "tables" / "training_history.csv"
    meta_path = run_dir / "checkpoints" / "best_compromise_meta.json"
    checkpoint_path = run_dir / "checkpoints" / "best_compromise.ckpt"
    for p in (history_path, meta_path, checkpoint_path):
        assert_val_only_path(p)
    hist = pd.read_csv(history_path)
    assert_no_test_columns(hist, str(history_path))
    missing = sorted(required_history - set(hist.columns))
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    preflight_rows.append({
        "site": site,
        "variant": variant,
        "run_dir": str(run_dir),
        "history_exists": history_path.exists(),
        "checkpoint_exists": checkpoint_path.exists(),
        "history_rows": len(hist),
        "unique_val_n": "|".join(map(str, sorted(hist["val_n"].dropna().unique()))),
        "missing_required_columns": "|".join(missing),
        "recorded_best_step": int(meta["step"]),
        "recorded_best_score": float(meta["article_compromise_score"]),
    })

preflight = pd.DataFrame(preflight_rows)
display(preflight)
assert preflight["history_exists"].all()
assert preflight["checkpoint_exists"].all()
assert (preflight["missing_required_columns"] == "").all()
preflight.to_csv(TABLE_DIR / "00_preflight_val_only.csv", index=False)
print("PASS: six VAL histories and retained checkpoint files were found; no TEST path/column was loaded.", flush=True)

In [ ]:
def selection_score(frame: pd.DataFrame, params: dict = BASE) -> np.ndarray:
    mae = frame["mae"].to_numpy(float)
    rmse = frame["rmse"].to_numpy(float)
    r2 = frame["r2"].to_numpy(float)
    bias = frame["bias"].to_numpy(float)
    slope = frame["slope"].to_numpy(float)
    sd = frame["std_ratio"].to_numpy(float)
    return (
        mae
        + params["w_rmse"] * rmse
        + params["w_slope"] * np.maximum(0.0, params["slope_floor"] - slope)
        + params["w_sd"] * np.abs(1.0 - sd)
        + params["w_bias"] * np.minimum(np.abs(bias), params["bias_cap"])
        + params["w_r2"] * np.maximum(0.0, params["r2_floor"] - r2)
    )


def add_objectives(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["abs_bias"] = out["bias"].abs()
    out["sd_gap"] = (1.0 - out["std_ratio"]).abs()
    out["slope_gap"] = (1.0 - out["slope"]).abs()
    out["r2_gap"] = 1.0 - out["r2"]
    return out


def pareto_mask(frame: pd.DataFrame, objectives=("mae", "abs_bias", "sd_gap", "slope_gap")) -> np.ndarray:
    values = frame.loc[:, objectives].to_numpy(float)
    finite = np.isfinite(values).all(axis=1)
    keep = np.zeros(len(frame), dtype=bool)
    for i in np.where(finite)[0]:
        dominated = np.any(
            np.all(values[finite] <= values[i], axis=1)
            & np.any(values[finite] < values[i], axis=1)
        )
        keep[i] = not dominated
    return keep


def add_mean_rank(frame: pd.DataFrame) -> pd.DataFrame:
    out = add_objectives(frame)
    rank_cols = ["mae", "rmse", "abs_bias", "sd_gap", "slope_gap", "r2_gap"]
    ranks = out[rank_cols].rank(method="average", ascending=True)
    out["mean_rank"] = ranks.mean(axis=1)
    return out


def selected_row(frame: pd.DataFrame, criterion: str) -> pd.Series:
    finite = frame[np.isfinite(frame[criterion])]
    if finite.empty:
        raise ValueError(f"No finite candidate for {criterion}")
    return finite.loc[finite[criterion].idxmin()]

In [ ]:
candidate_frames = []
reproduction_rows = []

for (site, variant), run_dir in RUNS.items():
    history = pd.read_csv(run_dir / "tables" / "training_history.csv")
    assert_no_test_columns(history, str(run_dir))
    meta = json.loads((run_dir / "checkpoints" / "best_compromise_meta.json").read_text(encoding="utf-8"))

    keep = history[[
        "step", "cycle", "val_n", "val_mae", "val_rmse", "val_r2",
        "val_bias", "val_slope", "val_std_ratio",
        "val_article_compromise_score", "val_checkpoint_eligible",
    ]].copy()
    keep = keep.rename(columns={c: c.removeprefix("val_") for c in keep.columns if c.startswith("val_")})
    keep.insert(0, "variant", variant)
    keep.insert(0, "site", site)
    keep["run_dir"] = str(run_dir)
    keep["recorded_retained"] = keep["step"].astype(int).eq(int(meta["step"]))
    keep["recomputed_score"] = selection_score(keep)
    keep["score_abs_difference"] = (keep["recomputed_score"] - keep["article_compromise_score"]).abs()
    keep = add_mean_rank(keep)
    keep["pareto_4d"] = pareto_mask(keep)
    candidate_frames.append(keep)

    retained = keep.loc[keep["recorded_retained"]]
    if len(retained) != 1:
        raise AssertionError(f"Expected exactly one retained row for {site}/{variant}; found {len(retained)}")
    reproduction_rows.append({
        "site": site,
        "variant": variant,
        "recorded_step": int(meta["step"]),
        "equation_selected_step": int(selected_row(keep, "recomputed_score")["step"]),
        "max_score_abs_difference": float(keep["score_abs_difference"].max()),
        "retained_is_pareto_4d": bool(retained["pareto_4d"].iloc[0]),
    })

candidates = pd.concat(candidate_frames, ignore_index=True)
reproduction = pd.DataFrame(reproduction_rows)
display(reproduction)

assert reproduction["max_score_abs_difference"].max() < 1e-8, "Printed equation does not reproduce stored score."
assert (reproduction["recorded_step"] == reproduction["equation_selected_step"]).all(), "Equation-selected step differs from recorded best-compromise step."

candidates.to_csv(TABLE_DIR / "01_all_recorded_val_evaluations.csv", index=False)
reproduction.to_csv(TABLE_DIR / "02_score_reproduction_and_pareto.csv", index=False)
print("PASS: the manuscript equation reproduces the stored validation score and retained step for all six runs.", flush=True)

In [ ]:
selector_rows = []
for (site, variant), group in candidates.groupby(["site", "variant"], sort=False):
    baseline = selected_row(group, "recomputed_score")
    mae_row = selected_row(group, "mae")
    rmse_row = selected_row(group, "rmse")
    rank_row = selected_row(group, "mean_rank")
    selector_rows.append({
        "site": site,
        "variant": variant,
        "n_candidates": len(group),
        "baseline_step": int(baseline["step"]),
        "mae_only_step": int(mae_row["step"]),
        "rmse_only_step": int(rmse_row["step"]),
        "mean_rank_step": int(rank_row["step"]),
        "same_step_mae": int(baseline["step"]) == int(mae_row["step"]),
        "same_step_rmse": int(baseline["step"]) == int(rmse_row["step"]),
        "same_step_mean_rank": int(baseline["step"]) == int(rank_row["step"]),
        "baseline_pareto_4d": bool(baseline["pareto_4d"]),
        "n_pareto_4d": int(group["pareto_4d"].sum()),
    })

selector_comparison = pd.DataFrame(selector_rows)
display(selector_comparison)
selector_comparison.to_csv(TABLE_DIR / "03_within_run_alternative_selectors.csv", index=False)

In [ ]:
def draw_params(rng: np.random.Generator, fraction: float) -> dict:
    p = dict(BASE)
    for key in ("w_rmse", "w_slope", "w_sd", "w_bias", "w_r2"):
        p[key] = BASE[key] * rng.uniform(1.0 - fraction, 1.0 + fraction)
    p["slope_floor"] = np.clip(BASE["slope_floor"] * rng.uniform(1.0 - fraction, 1.0 + fraction), 0.0, 1.0)
    p["r2_floor"] = np.clip(BASE["r2_floor"] * rng.uniform(1.0 - fraction, 1.0 + fraction), -1.0, 1.0)
    p["bias_cap"] = max(1e-6, BASE["bias_cap"] * rng.uniform(1.0 - fraction, 1.0 + fraction))
    return p


def sensitivity_for_group(group: pd.DataFrame, fraction: float, n_scenarios: int, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    baseline_scores = selection_score(group, BASE)
    baseline_order = pd.Series(baseline_scores).rank(method="average").to_numpy()
    baseline_idx = int(np.nanargmin(baseline_scores))
    selections = []
    taus = []
    for scenario in range(n_scenarios):
        params = draw_params(rng, fraction)
        scores = selection_score(group, params)
        idx = int(np.nanargmin(scores))
        order = pd.Series(scores).rank(method="average").to_numpy()
        tau = pd.Series(baseline_order).corr(pd.Series(order), method="kendall")
        selections.append(idx)
        taus.append(float(tau))
    freq = pd.Series(selections).value_counts().rename_axis("row_index").reset_index(name="count")
    freq["frequency"] = freq["count"] / n_scenarios
    freq["step"] = freq["row_index"].map(lambda i: int(group.iloc[int(i)]["step"]))
    summary = pd.DataFrame([{
        "fraction": fraction,
        "n_scenarios": n_scenarios,
        "baseline_step": int(group.iloc[baseline_idx]["step"]),
        "same_exact_step_frequency": float((np.asarray(selections) == baseline_idx).mean()),
        "median_kendall_tau": float(np.nanmedian(taus)),
        "q05_kendall_tau": float(np.nanquantile(taus, 0.05)),
        "n_distinct_selected_steps": int(pd.Series(selections).nunique()),
    }])
    return summary, freq


sens_summaries = []
sens_frequencies = []
for group_number, ((site, variant), group) in enumerate(candidates.groupby(["site", "variant"], sort=False)):
    group = group.reset_index(drop=True)
    for fraction in (0.25, 0.50):
        summary, freq = sensitivity_for_group(
            group, fraction=fraction, n_scenarios=N_SENSITIVITY,
            seed=RANDOM_SEED + group_number * 100 + int(fraction * 100),
        )
        summary.insert(0, "variant", variant)
        summary.insert(0, "site", site)
        freq.insert(0, "fraction", fraction)
        freq.insert(0, "variant", variant)
        freq.insert(0, "site", site)
        sens_summaries.append(summary)
        sens_frequencies.append(freq)

checkpoint_sensitivity = pd.concat(sens_summaries, ignore_index=True)
checkpoint_selection_frequencies = pd.concat(sens_frequencies, ignore_index=True)
display(checkpoint_sensitivity)

checkpoint_sensitivity.to_csv(TABLE_DIR / "04_within_run_weight_threshold_sensitivity.csv", index=False)
checkpoint_selection_frequencies.to_csv(TABLE_DIR / "05_within_run_selected_step_frequencies.csv", index=False)

In [ ]:
common = pd.read_csv(COMMON_SUPPORT_METRICS)
assert_no_test_columns(common, str(COMMON_SUPPORT_METRICS))
common = common[common["variant"].str.startswith("Phase 2")].copy()
common["variant_short"] = np.where(common["variant"].str.contains("residual", case=False), "residual_only", "temporal")
common = add_mean_rank(common)
common["baseline_score"] = selection_score(common)
common["pareto_4d"] = False
for site, idx in common.groupby("forest").groups.items():
    common.loc[idx, "pareto_4d"] = pareto_mask(common.loc[idx])

display(common[["forest", "variant", "n", "mae", "rmse", "bias", "r2", "slope", "std_ratio", "baseline_score", "pareto_4d"]])
common.to_csv(TABLE_DIR / "06_common_support_phase2_metrics_and_score.csv", index=False)

common_selector_rows = []
for site, group in common.groupby("forest", sort=False):
    group = group.copy()
    baseline = selected_row(group, "baseline_score")
    mae_row = selected_row(group, "mae")
    rmse_row = selected_row(group, "rmse")
    rank_row = selected_row(group, "mean_rank")
    common_selector_rows.append({
        "site": site,
        "baseline_variant": baseline["variant_short"],
        "mae_only_variant": mae_row["variant_short"],
        "rmse_only_variant": rmse_row["variant_short"],
        "mean_rank_variant": rank_row["variant_short"],
        "residual_only_pareto": bool(group.loc[group["variant_short"].eq("residual_only"), "pareto_4d"].iloc[0]),
        "temporal_pareto": bool(group.loc[group["variant_short"].eq("temporal"), "pareto_4d"].iloc[0]),
    })

common_selectors = pd.DataFrame(common_selector_rows)
display(common_selectors)
common_selectors.to_csv(TABLE_DIR / "07_common_support_alternative_selectors.csv", index=False)

In [ ]:
variant_sensitivity_rows = []
variant_scenario_rows = []
for site_number, (site, group) in enumerate(common.groupby("forest", sort=False)):
    group = group.reset_index(drop=True)
    baseline_idx = int(np.argmin(selection_score(group, BASE)))
    baseline_variant = group.iloc[baseline_idx]["variant_short"]
    for fraction in (0.25, 0.50):
        rng = np.random.default_rng(RANDOM_SEED + 1000 + site_number * 100 + int(100 * fraction))
        selected = []
        for scenario in range(N_SENSITIVITY):
            params = draw_params(rng, fraction)
            scores = selection_score(group, params)
            idx = int(np.argmin(scores))
            selected.append(group.iloc[idx]["variant_short"])
            variant_scenario_rows.append({
                "site": site,
                "fraction": fraction,
                "scenario": scenario,
                "selected_variant": group.iloc[idx]["variant_short"],
            })
        counts = pd.Series(selected).value_counts(normalize=True)
        variant_sensitivity_rows.append({
            "site": site,
            "fraction": fraction,
            "n_scenarios": N_SENSITIVITY,
            "baseline_variant": baseline_variant,
            "same_variant_frequency": float((np.asarray(selected) == baseline_variant).mean()),
            "residual_only_frequency": float(counts.get("residual_only", 0.0)),
            "temporal_frequency": float(counts.get("temporal", 0.0)),
        })

variant_sensitivity = pd.DataFrame(variant_sensitivity_rows)
variant_scenarios = pd.DataFrame(variant_scenario_rows)
display(variant_sensitivity)
variant_sensitivity.to_csv(TABLE_DIR / "08_common_support_variant_sensitivity.csv", index=False)
variant_scenarios.to_csv(TABLE_DIR / "09_common_support_variant_scenarios.csv", index=False)

In [ ]:
site_order = ["Ifran", "Maamoura", "Agadir"]
colors = {"residual_only": "#0072B2", "temporal": "#D55E00"}
markers = {"residual_only": "o", "temporal": "s"}

fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.2), constrained_layout=True)
for ax, site in zip(axes, site_order):
    g = common[common["forest"].eq(site)]
    for _, row in g.iterrows():
        variant = row["variant_short"]
        ax.scatter(row["mae"], row["sd_gap"], s=75, color=colors[variant], marker=markers[variant], zorder=3)
        ax.annotate(variant.replace("_", " "), (row["mae"], row["sd_gap"]), xytext=(5, 5), textcoords="offset points", fontsize=8)
    ax.set_title(site, fontweight="bold")
    ax.set_xlabel("MAE (m)")
    ax.grid(alpha=0.25)
axes[0].set_ylabel(r"$|1-\mathrm{SD\ ratio}|$")
fig.suptitle("Selected Phase 2 variants on common VAL support", fontweight="bold")
fig.savefig(FIG_DIR / "C1_common_val_tradeoff.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "C1_common_val_tradeoff.png", dpi=300, bbox_inches="tight")
plt.show()

plot_df = variant_sensitivity.copy()
plot_df["range"] = plot_df["fraction"].map({0.25: "±25%", 0.50: "±50%"})
fig, ax = plt.subplots(figsize=(7.4, 3.6))
x = np.arange(len(site_order))
width = 0.34
for offset, label in zip((-width / 2, width / 2), ("±25%", "±50%")):
    values = [plot_df[(plot_df["site"].eq(site)) & (plot_df["range"].eq(label))]["same_variant_frequency"].iloc[0] for site in site_order]
    bars = ax.bar(x + offset, values, width, label=label)
    ax.bar_label(bars, labels=[f"{100*v:.1f}%" for v in values], padding=2, fontsize=8)
ax.set_xticks(x, site_order)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Baseline variant retained")
ax.set_title("Sensitivity to heuristic weights and thresholds", fontweight="bold")
ax.legend(frameon=False, ncol=2)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "C2_common_val_variant_sensitivity.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "C2_common_val_variant_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
lines = []
for _, row in variant_sensitivity.iterrows():
    lines.append(
        f"- {row['site']} ({int(row['fraction']*100)}% perturbation): "
        f"baseline variant retained in {100*row['same_variant_frequency']:.1f}% of scenarios."
    )

all_residual_baseline = common_selectors["baseline_variant"].eq("residual_only").all()
all_residual_25 = variant_sensitivity.loc[variant_sensitivity["fraction"].eq(0.25), "residual_only_frequency"].min()

if all_residual_baseline:
    conclusion = "On the common VAL support, the original heuristic selected residual-only Phase 2 in all three landscapes."
else:
    conclusion = "On the common VAL support, the original heuristic did not select the same Phase 2 variant in all landscapes."

safe_wording = (
    "The penalised multi-metric score was used solely as an internal VAL checkpoint-ranking heuristic. "
    "Its coefficients represent study-specific selection preferences rather than statistically estimated or universal weights. "
    + conclusion
)

display(Markdown("### Sensitivity summary\n" + "\n".join(lines) + "\n\n### Manuscript-safe statement\n> " + safe_wording))

summary = {
    "scope": "VAL only; TEST not loaded",
    "score_reproduced_for_all_runs": bool((reproduction["recorded_step"] == reproduction["equation_selected_step"]).all()),
    "all_recorded_checkpoints_pareto_4d": bool(reproduction["retained_is_pareto_4d"].all()),
    "common_support_baseline_residual_all_sites": bool(all_residual_baseline),
    "minimum_residual_only_frequency_under_25pct": float(all_residual_25),
    "safe_wording": safe_wording,
}
(OUT_DIR / "audit_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(OUT_DIR / "audit_summary.md").write_text(
    "# Phase 2 checkpoint-selection robustness audit\n\n"
    + "\n".join(lines)
    + "\n\n## Manuscript-safe statement\n\n"
    + safe_wording
    + "\n",
    encoding="utf-8",
)
print(f"Saved audit outputs to: {OUT_DIR}", flush=True)